### Import Hadoop setup function

In [ ]:
# Download hadoop_import.py from https://github.com/sharadgupta27/data-engineering/blob/main/Homework/hadoop_import.py
# and keep it in the same directory as this script. Then run this script to set up Hadoop for Windows.
import os
from hadoop_import import setup_hadoop_for_windows
setup_hadoop_for_windows()

winutils.exe already present at C:\hadoop\bin\winutils.exe
hadoop.dll already present at C:\hadoop\bin\hadoop.dll

HADOOP_HOME = C:\hadoop
Both winutils.exe and hadoop.dll are in place. Restart kernel if SparkSession was already created.


True

**Question 1**: Install Spark and PySpark

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [4]:
spark.version

'4.1.1'

Asnwer: **4.1.1**

------------------------------------------------------------

**Question 2**: Yellow November 2025

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [ ]:
# !ls -lh yellow_tripdata_2025-11.parquet

In [5]:
import pandas as pd

df_pd = pd.read_parquet("yellow_tripdata_2025-11.parquet")
df_pd.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,7,2025-11-01 00:13:25,2025-11-01 00:13:25,1.0,1.68,1.0,N,43,186,1,14.9,0.00,0.5,1.50,0.00,1.0,22.15,2.5,0.00,0.75
1,2,2025-11-01 00:49:07,2025-11-01 01:01:22,1.0,2.28,1.0,N,142,237,1,14.2,1.00,0.5,4.99,0.00,1.0,24.94,2.5,0.00,0.75
2,1,2025-11-01 00:07:19,2025-11-01 00:20:41,0.0,2.70,1.0,N,163,238,1,15.6,4.25,0.5,4.27,0.00,1.0,25.62,2.5,0.00,0.75
3,2,2025-11-01 00:00:00,2025-11-01 01:01:03,3.0,12.87,1.0,N,138,261,1,66.7,6.00,0.5,0.00,6.94,1.0,86.14,2.5,1.75,0.75
4,1,2025-11-01 00:18:50,2025-11-01 00:49:32,0.0,8.40,1.0,N,138,37,2,39.4,7.75,0.5,0.00,0.00,1.0,48.65,0.0,1.75,0.00


In [6]:
schema = types.StructType([
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True),
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('passenger_count', types.IntegerType(), True),
    types.StructField('trip_distance', types.FloatType(), True),
    types.StructField('fare_amount', types.FloatType(), True),
])

In [7]:
df = spark.read \
    .parquet('yellow_tripdata_2025-11.parquet', schema=schema)
    
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [8]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [9]:
df = df.repartition(4)

df.write.parquet('data/pq/yellow/2025/11/', mode='overwrite')

In [10]:
df = spark.read.parquet('data/pq/yellow/2025/11/')

In [11]:
!du -h data/pq/yellow

99M	data/pq/yellow/2025/11
99M	data/pq/yellow/2025
99M	data/pq/yellow


In [12]:
!ls -lh data/pq/yellow/2025/11

total 98M
-rw-r--r-- 1 gupta INTRANET+Group(513)   0 Mar  9 03:01 _SUCCESS
-rw-r--r-- 1 gupta INTRANET+Group(513) 25M Mar  9 03:01 part-00000-d721aaa4-babc-439f-bd40-ddd04e1a0cfc-c000.snappy.parquet
-rw-r--r-- 1 gupta INTRANET+Group(513) 25M Mar  9 03:01 part-00001-d721aaa4-babc-439f-bd40-ddd04e1a0cfc-c000.snappy.parquet
-rw-r--r-- 1 gupta INTRANET+Group(513) 25M Mar  9 03:01 part-00002-d721aaa4-babc-439f-bd40-ddd04e1a0cfc-c000.snappy.parquet
-rw-r--r-- 1 gupta INTRANET+Group(513) 25M Mar  9 03:01 part-00003-d721aaa4-babc-439f-bd40-ddd04e1a0cfc-c000.snappy.parquet


Answer: ~**100MB**

------------------------------------------------------------------

**Question 3**: How many taxi trips were there on November 15?

In [13]:
from pyspark.sql import functions as F

In [14]:
df \
    .withColumn('tpep_pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
    .filter("tpep_pickup_datetime = '2025-11-15'") \
    .count()

162604

In [15]:
df.createOrReplaceTempView('yellow_2025_11')

In [16]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    yellow_2025_11
WHERE
    to_date(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



Answer: **162604**

----------------------------------------------------------------------

**Question 4**: Longest trip for each day

In [17]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [18]:
df \
    .withColumn('duration_hours', (F.unix_timestamp(df.tpep_dropoff_datetime) - F.unix_timestamp(df.tpep_pickup_datetime))/3600) \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .groupBy('pickup_date') \
        .max('duration_hours') \
    .orderBy('max(duration_hours)', ascending=False) \
    .limit(10) \
    .show()

+-----------+-------------------+
|pickup_date|max(duration_hours)|
+-----------+-------------------+
| 2025-11-26|  90.64666666666666|
| 2025-11-27|  76.94833333333334|
| 2025-11-03|  76.21388888888889|
| 2025-11-07|  69.28861111111111|
| 2025-11-18|  67.08055555555555|
| 2025-11-22|  63.36833333333333|
| 2025-11-01| 56.382222222222225|
| 2025-11-05| 42.720555555555556|
| 2025-11-06| 41.614444444444445|
| 2025-11-24| 38.074444444444445|
+-----------+-------------------+



In [19]:
spark.sql("""
SELECT
    to_date(tpep_pickup_datetime) AS pickup_date,
    MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS duration_hours
FROM 
    yellow_2025_11
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 10;
""").show()

+-----------+------------------+
|pickup_date|    duration_hours|
+-----------+------------------+
| 2025-11-26| 90.64666666666666|
| 2025-11-27| 76.94833333333334|
| 2025-11-03| 76.21388888888889|
| 2025-11-07| 69.28861111111111|
| 2025-11-18| 67.08055555555555|
| 2025-11-22| 63.36833333333333|
| 2025-11-01|56.382222222222225|
| 2025-11-05|42.720555555555556|
| 2025-11-06|41.614444444444445|
| 2025-11-24|38.074444444444445|
+-----------+------------------+



Answer: **90.6 hours**

-------------------------------------------------------------------

**Question 5**: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040
- 8080

Answer: **4040**

-----------------------------------------------------------------------

**Question 6**: Least frequent `pickup location zone`

In [20]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 03:02:59--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:21ca:3c00:b:20a5:b140:21, 2600:9000:21ca:4e00:b:20a5:b140:21, 2600:9000:21ca:6a00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:21ca:3c00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv.1'

     0K .......... ..                                         100%  231M=0s

2026-03-09 03:02:59 (231 MB/s) - 'taxi_zone_lookup.csv.1' saved [12331/12331]



In [21]:
# Load zone lookup into a Spark temp view

df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('taxi_zone_lookup.csv')

df_zones.show(5)
df_zones.createOrReplaceTempView('zones')

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [31]:
spark.sql("""
SELECT
    z.Zone,
    COUNT(1) AS pickup_count
FROM
    yellow_2025_11 t
    JOIN zones z ON t.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY pickup_count ASC, z.Zone ASC
LIMIT 5;
""").show(truncate=False)

+---------------------------------------------+------------+
|Zone                                         |pickup_count|
+---------------------------------------------+------------+
|Arden Heights                                |1           |
|Eltingville/Annadale/Prince's Bay            |1           |
|Governor's Island/Ellis Island/Liberty Island|1           |
|Port Richmond                                |3           |
|Great Kills                                  |4           |
+---------------------------------------------+------------+



Answer: **Arden Heights**